# **Text-to-SQL Query Helper Tool**

## **Project Overview**

`This project builds a natural language-to-SQL generation tool using a local, lightweight Large Language Model (TinyLlama-1.1B-Chat). By leveraging the transformers library alongside langchain and langchain_huggingface, the notebook sets up a pipeline that takes a plain-English request (e.g., "Extract Top 5 Avg Salary from Employee Table") and automatically outputs the corresponding SQL code. It is a great starting point for building offline, AI-assisted database querying tools.`

## **Install Required Dependencies**

In [13]:
# Install required libraries, ensuring ecosystem compatibility
!pip install -q --upgrade transformers langchain langchain-core langchain-huggingface huggingface-hub accelerate

## **Authenticate with Hugging Face (Securely)**

In [14]:
from google.colab import userdata
from huggingface_hub import login

# 1. Fetch your hidden key using the exact name you defined in the Secrets tab
hf_token = userdata.get('API_Key')

# 2. Log in using the retrieved token
if hf_token:
    login(hf_token)
else:
    print("Secret key 'API_Key' not found. Please check your Colab Secrets settings.")

## **Initialize Model and Tokenizer Pipeline**

In [15]:
from transformers import AutoTokenizer, pipeline
import torch

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Set up text generation pipeline with deprecations fixed and strict formatting
text_pipeline = pipeline(
    "text-generation",
    model=model_id,
    tokenizer=tokenizer,
    dtype=torch.bfloat16,    # Replaced deprecated torch_dtype
    device_map="auto",
    max_new_tokens=150,      # Reduced to prevent excessive rambling
    do_sample=False,         # Use greedy decoding for precise SQL generation
    return_full_text=False   # Prevents the prompt from being printed in the output
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

## **Setup LangChain LLM Wrapper**

In [16]:
from langchain_huggingface import HuggingFacePipeline

# Wrap the Hugging Face pipeline in LangChain
llm = HuggingFacePipeline(pipeline=text_pipeline)

## **Define Prompt Template and Build Chain**

In [17]:
from langchain_core.prompts import PromptTemplate

# Using TinyLlama's specific chat template structure for much better instruction following
template = """<|system|>
You are an expert SQL assistant. Generate only the raw SQL query based on the user's text. Do not provide explanations, markdown formatting, or additional examples.</s>
<|user|>
Create a SQL query snippet using the below text:
{text}</s>
<|assistant|>
"""

prompt = PromptTemplate(template=template, input_variables=["text"])

# Modern LCEL (LangChain Expression Language) syntax
llm_chain = prompt | llm

## **Execute the Query Generation Chain**

In [18]:
# Prompt the user to enter their plain-English request
text_input = input("Enter your request for the SQL query: ")

# Invoke the chain and print the generated SQL
response = llm_chain.invoke({"text": text_input})

print("\nGenerated SQL Query:\n")
print(response.strip())

Enter your request for the SQL query: Extract Top 5 Avg Salary from Employee table


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Generated SQL Query:

SELECT TOP 5 AVG(Salary) AS AvgSalary
FROM Employee
ORDER BY Salary DESC;


## **Conclusion**

**This Text-to-SQL pipeline cleanly integrates the TinyLlama model with LangChain, utilizing the model's native conversational template for accurate instruction following. By securely managing API credentials and applying strict generation parameters—such as disabling sampling and enforcing a precise prompt structure—the tool reliably outputs isolated, executable SQL queries without generating conversational filler or hallucinations. This robust architecture forms a strong, production-ready foundation for automated, natural-language-driven database querying.**